# Connecting to MRS Hive from a ModelArts Notebook (Kerberos-secured cluster)

In this notebook (same VPC as MRS, with public internet access) we go through:
**environment setup -> network probing -> krb5 configuration -> kinit -> Kerberos
connection to HiveServer2 -> verification queries -> a general query helper**.

> ✅ **Verified end-to-end on a real ModelArts Notebook on 2026-08-19** (CPU image /
> ma-user without root): cell 7 prints `databases: ['default', 'mrs_system']` and
> `breast_cancer row count: 569`.

The recipe comes from hands-on troubleshooting on the master1 node on 2026-08-19.
The single authoritative source for cluster facts is `hive_export/MRS_RUN.md §0`
(Chinese); the decision record is `docs/adr/0002`:

| Key fact | Value |
|---|---|
| HiveServer2 | `10.0.0.15:21066` (internal IP, binary transport) |
| SPN middle part | `hadoop.252a63ec_2c90_4b5a_b4d7_17a3077b1cb8.com` (`hadoop.` + system domain, all lowercase; the `haddop_` variant does **not** exist in the KDC) |
| KDC port | **21732** (not the open-source default 88; master1=10.0.0.15, master2=10.0.0.51) |
| SASL QOP | **auth-conf** (encrypted wrapping layer) -> the cyrus `sasl` pip package is mandatory; pure-sasl+pykerberos was measured to fail at the encryption stage |

**Prerequisites**:
1. Security group allows notebook -> `10.0.0.15:21066` and `10.0.0.15/10.0.0.51:21732` (TCP+UDP);
2. the notebook can reach the public internet (pip installs dependencies);
3. any of root / passwordless sudo / neither (ma-user) works — cell 3 auto-adapts.

> kinit credentials: this notebook prompts for the hhx password interactively via
> `getpass` (no plaintext left in code); the non-interactive keytab alternative is
> described at the end of §8.
>
> _中文原版: `modelarts_hive_conn.ipynb`（同提交同步，见 ADR-0001）。_


In [ ]:
# ================== 1. Connection settings (measured values; edit here for another cluster) ==================
HIVE_HOST = "10.0.0.15"    # HiveServer2 internal IP (master1)
HIVE_PORT = 21066          # HiveServer2 Thrift port
DATABASE  = "default"
USERNAME  = "hhx"          # MRS business user

REALM    = "252A63EC_2C90_4B5A_B4D7_17A3077B1CB8.COM"  # MRS system domain (Realm)
SPN_HOST = "hadoop." + REALM.lower()   # measured-correct SPN middle part (haddop_ variant is wrong)

KDC_HOSTS = ["10.0.0.15", "10.0.0.51"]  # list both masters for fault tolerance
KDC_PORT  = 21732                        # Huawei MRS dedicated KDC port, NOT 88!

print("principal =", f"hive/{SPN_HOST}@{REALM}")


In [ ]:
# ================== 2. Network probing (a missing security-group rule surfaces here fast) ==================
import socket

def probe(host, port, name, timeout=5):
    s = socket.socket(); s.settimeout(timeout)
    try:
        s.connect((host, port)); print(f"[OK]   {name} {host}:{port} reachable")
        return True
    except Exception as e:
        print(f"[FAIL] {name} {host}:{port} unreachable: {e}")
        return False
    finally:
        s.close()

net_ok = probe(HIVE_HOST, HIVE_PORT, "HiveServer2")
for k in KDC_HOSTS:
    net_ok &= probe(k, KDC_PORT, "KDC")

assert net_ok, (
    "Network unreachable: confirm the notebook is in the same VPC as MRS, and the\n"
    "security group allows 21066 and 21732 (TCP+UDP); 21066 alone is not enough - kinit also needs the KDC on 21732."
)


In [ ]:
# ================== 3. Environment setup (idempotent; auto-adapts to three environments; live progress) ==================
# Goal: kinit binary + cyrus sasl (with the GSSAPI plugin in place) + pure-python pyhive etc.
#   * The cluster enforces qop=auth-conf, so cyrus sasl is mandatory; pure-sasl+pykerberos
#     was measured to fail at the encrypted-wrap stage with "Invalid token was supplied"
#     (see ADR-0002).
# Adaptation order (the [env] line tells you which branch fired):
#   A. root             -> apt install gcc/g++/krb5-user/headers + GSSAPI plugin, pip-build sasl
#   B. ma-user + sudo   -> same as A, with sudo -n in front of apt
#   C. no root (common) -> conda-forge prebuilt: krb5 (ships kinit) + sasl, no compiler needed
import collections, importlib, os, re, shutil, subprocess, sys, threading, time

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

# ---- live-progress runner: print key lines immediately (with elapsed time),
#      heartbeat during quiet periods, replay tail output on failure ----
_BAR = re.compile(r"^\W*\[\W*\d+%\W*\]\W*$")        # apt's "[ 12%]" progress bars (noise)
_HOT = re.compile(r"solving|collecting|downloading|extracting|preparing|executing|"
                  r"transaction|unpacking|setting up|processing|fetched|^get|^hit|"
                  r"building wheel|successfully|installed|nothing to do|all requested|"
                  r"error|fail|conflict|warn", re.I)     # progress/result lines worth showing

def run_stream(cmd, note=None, heartbeat=20):
    """Stream an external command. Return code 0 = success; on failure replay the last 40 lines."""
    if note: print(f"[run] {note}", flush=True)
    t0, tail = time.time(), collections.deque(maxlen=40)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    stop = threading.Event()
    def _beat():                                          # prove "still alive" during long silences
        quiet = time.time()
        while not stop.wait(2):
            if time.time() - quiet >= heartbeat:
                print(f"   ... {int(time.time()-t0)}s still running ({cmd[0]} silent, that is normal)", flush=True)
                quiet = time.time()
    th = threading.Thread(target=_beat, daemon=True); th.start()
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line)
        if line and not _BAR.match(line) and _HOT.search(line):
            print(f"[{int(time.time()-t0):>3}s] {line}", flush=True)
    rc = proc.wait(); stop.set(); th.join(timeout=1)
    if rc != 0:
        print("---- tail of command output (up to 40 lines) ----")
        print("\n".join(t for t in tail if t.strip()) or "(no output)")
    return rc

# conda's kinit lives in sys.prefix/bin: after a kernel restart PATH may not include
# it, so prepend it here — otherwise need_kinit is misdetected even though deps are
# already installed, triggering a pointless conda solve (~5 minutes measured here)
if os.path.isfile(os.path.join(sys.prefix, "bin", "kinit")):
    os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
need_kinit, need_sasl = shutil.which("kinit") is None, not have("sasl")

# --- 3.1 system layer ---
if need_kinit or need_sasl:
    apt, env_name = None, "no root (conda branch)"
    if os.geteuid() == 0:
        apt, env_name = ["apt-get"], "root"
    else:
        sudo_ok = subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0
        if sudo_ok:
            apt, env_name = ["sudo", "-n", "apt-get"], "ma-user + passwordless sudo"
    print(f"[env] {env_name}", flush=True)

    if apt is not None:
        # libsasl2-modules-gssapi-mit / libsasl2-modules = cyrus GSSAPI plugins (required!)
        if run_stream(apt + ["update"], "apt-get update") != 0:
            raise SystemExit("[FAIL] apt-get update failed")
        if run_stream(apt + ["install", "-y", "gcc", "g++", "krb5-user",
                             "libkrb5-dev", "libsasl2-dev",
                             "libsasl2-modules-gssapi-mit", "libsasl2-modules"],
                      "apt install toolchain + krb5 + cyrus sasl (first run ~1-2 min)") != 0:
            raise SystemExit("[FAIL] apt install failed")
    else:
        # no root: ModelArts ships anaconda; conda-forge has prebuilt krb5 and sasl
        conda = shutil.which("conda")
        assert conda, "[FAIL] conda not found — please report this error to the maintainers"
        print("[env] no root — using the conda-forge prebuilt path", flush=True)
        # --prefix sys.prefix: install explicitly into the current kernel env, not base
        # --override-channels: use only the channels given here — if the instance's
        #   condarc points at dead mirror channels (e.g. TUNA anaconda/pkgs/free,
        #   no longer synced, 404), they would fail too without this flag
        base = [conda, "install", "-y", "--override-channels", "--prefix", sys.prefix]
        attempts = [
            (base + ["-c", "conda-forge", "krb5", "sasl"],
             "conda install krb5 + sasl (conda-forge, bypassing dead mirror channels; solve+download 1-3 min)"),
            (base + ["-c", "https://conda.anaconda.org/conda-forge", "krb5", "sasl"],
             "conda retry (direct official conda-forge, may be slower)"),
        ]
        rc = 1
        for cmd, note in attempts:
            rc = run_stream(cmd, note)
            if rc == 0:
                break
        if rc != 0:
            raise SystemExit("[FAIL] conda install failed on both sources (instance mirrors down?); "
                             "fallback: offline wheels via OBS, or report the output above to maintainers")
        # conda's kinit lives in $CONDA_PREFIX/bin — put it on PATH for cell 5
        os.environ["PATH"] = os.path.join(sys.prefix, "bin") + os.pathsep + os.environ["PATH"]
        print("[tip] If the 3.3 self-check below fails to import (kernel unaware of freshly installed conda "
              "packages), restart the kernel and re-run cells 1, 3, 4 — installed pieces are skipped automatically")
else:
    print("[env] system dependencies already in place (kinit + sasl), skipping installation")

# --- 3.2 python packages (pure python, pip is enough) ---
PIP_PKGS = [p for p, m in (("pyhive", "pyhive"), ("thrift", "thrift"),
                           ("thrift-sasl", "thrift_sasl"), ("sasl", "sasl"))
            if not have(m)]
if PIP_PKGS:
    if run_stream([sys.executable, "-m", "pip", "install"] + PIP_PKGS,
                  f"pip install {PIP_PKGS}") != 0:
        raise SystemExit("[FAIL] pip install failed")

# --- 3.3 self-check: imports + cyrus GSSAPI plugin in place (hard prerequisite for auth-conf) ---
# Note: the cyrus sasl package has no "list mechanisms" API (available_mechs belongs
# to pure-sasl; using it raises AttributeError). Use a functional probe instead:
# actually init + start GSSAPI once — the same code path cell 6 uses at connect time
# (pyhive.get_sasl_client -> setAttr+init; thrift_sasl.open -> start):
#   start succeeds                                 -> plugin in place (and a ticket exists)
#   "No worthy mechs" / "No mechanism available"   -> plugin missing (deps incomplete, fatal)
#   GSSAPI credential error (no ticket, etc.)      -> plugin in place; works after cell 5 kinit
import glob
from pyhive import hive
from pyhive.hive import get_installed_sasl
import thrift_sasl
import sasl as cyrus_sasl

_p = cyrus_sasl.Client()
_p.setAttr("host", SPN_HOST)          # SPN middle part from cell 1, as the SASL-layer parameter
_p.setAttr("service", "hive")
assert _p.init(), f"cyrus sasl init failed: {_p.getError()!r}"
_ok, _mech, _resp = _p.start("GSSAPI")
_err = _p.getError()
_err = _err.decode("utf-8", "replace") if isinstance(_err, bytes) else (_err or "")
if _ok:
    print("[OK] python dependencies ready; GSSAPI plugin usable; kinit =", shutil.which("kinit"))
elif "worthy mechs" in _err.lower() or "no mechanism available" in _err.lower():
    for _pat in (os.path.join(sys.prefix, "lib*", "sasl2", "*"),
                 "/usr/lib/*/sasl2/*", "/usr/lib64/sasl2/*"):
        for _h in glob.glob(_pat):
            if "gssapi" in os.path.basename(_h).lower():
                print("  gssapi plugin file:", _h)
    raise SystemExit(
        f"cyrus sasl is missing the GSSAPI plugin ({_err})\n"
        "root/sudo env: check that libsasl2-modules-gssapi-mit got installed;\n"
        "conda env: send the output of !ls $CONDA_PREFIX/lib/sasl2/ to the maintainers")
else:
    print("[OK] python dependencies ready; GSSAPI plugin in place (no ticket yet — effective "
          f"after kinit in cell 5; probe info: {_err.splitlines()[0] if _err else '-'})")
    print("kinit =", shutil.which("kinit"))


In [ ]:
# ================== 4. Generate krb5.conf and make it effective ==================
# dns_canonicalize_hostname=false is the key: the SPN middle part hadoop.xxx is a
# "fake domain" that does not exist in DNS — the Kerberos client must be stopped
# from resolving it, so it is used verbatim as the SPN.
# udp_preference_limit=1 forces AS/TGS requests onto TCP — matching the TCP port probed in cell 2.
import os
from pathlib import Path

KRB5_FILE = Path.cwd() / "krb5.conf"
lines = [
    "[libdefaults]",
    f"    default_realm = {REALM}",
    "    dns_canonicalize_hostname = false",   # <- key
    "    rdns = false",
    "    udp_preference_limit = 1",
    "",
    "[realms]",
    f"    {REALM} = {{",
    *[f"        kdc = {h}:{KDC_PORT}" for h in KDC_HOSTS],
    f"        admin_server = {KDC_HOSTS[0]}:{KDC_PORT}",
    "    }",
    "",
    "[domain_realm]",
    f"    .{REALM.lower()} = {REALM}",
    f"    {SPN_HOST} = {REALM}",
    f"    .{SPN_HOST} = {REALM}",
    "",
]
KRB5_FILE.write_text("\n".join(lines), encoding="utf-8")
os.environ["KRB5_CONFIG"] = str(KRB5_FILE)   # both kinit and cyrus-sasl read this later
print(f"[OK] generated {KRB5_FILE} and set KRB5_CONFIG\n")
print("\n".join(lines))


In [ ]:
# ================== 5. kinit to obtain the user ticket (TGT, valid 24h) ==================
# Skip if a valid ticket exists; otherwise prompt for the password (getpass — no
# plaintext left in code). kinit may come from apt (krb5-user, /usr/bin) or conda
# (krb5, $CONDA_PREFIX/bin) — adapt automatically.
import getpass, shutil, subprocess

KINIT = shutil.which("kinit") or os.path.join(sys.prefix, "bin", "kinit")
KLIST = shutil.which("klist") or os.path.join(sys.prefix, "bin", "klist")

def _run(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True,
                          env={**os.environ, "KRB5_CONFIG": str(KRB5_FILE)}, **kw)

r = _run([KLIST])
if r.returncode == 0 and "krbtgt" in r.stdout:
    print("[OK] a valid ticket already exists, skipping kinit:")
    print("\n".join(r.stdout.splitlines()[:4]))
else:
    principal = f"{USERNAME}@{REALM}"
    pw = getpass.getpass(f"Password for {principal}: ")
    r = _run([KINIT, principal], input=pw + "\n")
    assert r.returncode == 0, f"[FAIL] kinit failed (wrong password / KDC unreachable?): {r.stderr.strip()}"
    print(f"[OK] kinit succeeded: {principal}")


In [ ]:
# ================== 6. Connect to HiveServer2 (core: decouple TCP address from SPN) ==================
# With auth=KERBEROS pyhive uses the TCP host directly as the SPN host -> guaranteed
# mismatch (it would ask the KDC for hive/10.0.0.15@REALM, but the cluster registers
# a fixed-string SPN). The official escape hatch is thrift_transport=...: the TCP
# layer connects to the internal IP while the SASL-layer host carries the SPN middle part.
# get_installed_sasl automatically prefers the cyrus sasl package once installed
# (mandatory for qop=auth-conf).
from thrift.transport import TSocket

def make_transport():
    tcp = TSocket.TSocket(HIVE_HOST, HIVE_PORT)
    tcp.setTimeout(30000)
    sasl_factory = lambda: get_installed_sasl(
        host=SPN_HOST, sasl_auth="GSSAPI", service="hive")
    return thrift_sasl.TSaslClientTransport(sasl_factory, "GSSAPI", tcp)

PRINCIPAL = f"hive/{SPN_HOST}@{REALM}"
print("Trying SPN:", PRINCIPAL)
conn = hive.connect(thrift_transport=make_transport(),
                    database=DATABASE, username=USERNAME)
print("[OK] connected! Effective SPN =", PRINCIPAL)


In [ ]:
# ================== 7. Verification queries ==================
cur = conn.cursor()

cur.execute("SHOW DATABASES")
print("databases:", [r[0] for r in cur.fetchall()])

cur.execute("SHOW TABLES")
tables = [r[0] for r in cur.fetchall()]
print("tables   :", tables)

if "breast_cancer" in tables:
    cur.execute("SELECT COUNT(*) FROM breast_cancer")
    print("breast_cancer row count:", cur.fetchone()[0])
cur.close()


## 8. General query helper run_query()

Run any SQL later via `run_query("...")`.

**Troubleshooting quick reference** (full version in `hive_export/MRS_RUN.md §5`, Chinese):

| Symptom | Fix |
|---|---|
| Cell 3 `Permission denied (apt lock)` | Not a bug — the cell falls back to the sudo/conda branch automatically; if it still fails, send the `[env]` line and the error to the maintainers |
| `Invalid token was supplied` | The pure-sasl+pykerberos path was taken -> re-run cell 3, confirm the self-check prints "GSSAPI plugin in place/usable", then restart the kernel and run from the top |
| KDC probe fails | Allow **21732** (TCP+UDP) in the security group, not 88 |
| kinit says wrong password | MRS Manager -> System -> User Management -> reset the hhx password |
| `import pandas` raises `numpy.dtype size changed` | The image puts `~/modelarts-dev/modelarts-sdk` (a pandas source tree built against numpy 2.x) on sys.path, shadowing the properly installed pandas -> the code cell below removes it and cleans up automatically at the top; if a hand-written cell hits the same error, re-run this cell |

**Ticket expires after 24h / instance restarted**: re-run cell 5 (re-enter the password); **after a kernel restart** cells 1, 3, 4 must be re-run (PATH/KRB5_CONFIG etc. live only in memory).

**Non-interactive keytab alternative**: MRS Manager -> System -> User Management -> hhx -> More -> Download Credentials to get `user.keytab`, upload it next to the notebook, then replace cell 5's kinit branch with:

```python
r = _run([KINIT, "-kt", "user.keytab", f"{USERNAME}@{REALM}"])
```

**Remember to close the connection when done**: `conn.close()` (included at the end of the example below).


In [ ]:
# ================== 8. General query helper ==================
# Pitfall (some ModelArts images): ~/modelarts-dev/modelarts-sdk contains a pandas
# source tree built against numpy 2.x and is put on sys.path, shadowing the properly
# installed pandas — on a numpy 1.x kernel importing it raises "numpy.dtype size
# changed" directly. Remove the entries first, then import.
# Measured (2026-08-19): this instance has two shadowing directories — besides
# modelarts-sdk and ma-cli there is /modelarts/tools/solution/advisor (pandas 2.3.2);
# after removing the first two, pandas loads from advisor, which happened to be
# binary-compatible with the kernel's numpy 2.0.2 and thus worked; rely on the
# path echoed below to confirm where it actually came from.
import subprocess, sys

_dev = [p for p in sys.path if "modelarts-dev" in p or "modelarts-sdk" in p]
for p in _dev:
    sys.path.remove(p)
if _dev:
    print("[fix] removed development directories from sys.path (they shadow the real pandas):", ", ".join(_dev))
# A failed import leaves half-initialized modules in sys.modules; purge them so the
# retry works in place (no kernel restart needed)
for m in [m for m in list(sys.modules) if m == "pandas" or m.startswith("pandas.")]:
    del sys.modules[m]

import numpy as np
try:
    import pandas as pd
except ModuleNotFoundError:                      # only install if the env truly lacks pandas
    subprocess.run([sys.executable, "-m", "pip", "install", "pandas"], check=True)
    import pandas as pd
print(f"[OK] pandas {pd.__version__} <- {pd.__file__}")
print(f"[OK] numpy  {np.__version__} <- {np.__file__}")

def run_query(sql, max_rows=20):
    """Execute SQL, print and return a DataFrame (display truncated to max_rows)."""
    cur = conn.cursor()
    try:
        cur.execute(sql)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchall()
    finally:
        cur.close()
    df = pd.DataFrame(rows, columns=cols)
    with pd.option_context("display.max_rows", max_rows):
        display(df)
    return df

# Example: fetch 5 rows
_ = run_query("SELECT * FROM breast_cancer LIMIT 5")

# Close the connection when done:
# conn.close(); print("closed")
